# EYEWAZ — train your Urdu voice (clean, 3 cells)

**KAGGLE (recommended):** New Notebook → right panel **Accelerator: GPU T4 x2**,
**Internet: On** → **Add Input → Upload `dataset-male.zip`** → **Run All**.

**COLAB:** Runtime → GPU; `dataset-male.zip` in Google Drive/eyewaz/ → Run All.

⚠️ **Keep the tab open.** On Kaggle, when the .onnx is exported, hit **Save Version**
so the result persists (that's how the file got lost last time).


## 1 · Setup  (install + dataset + checkpoint — ~5 min)


In [ ]:
import os, glob, shutil, subprocess
KAGGLE = os.path.isdir('/kaggle/working'); WORK = '/kaggle/working' if KAGGLE else '/content'
VOICE = 'male'      # <- change for other voices/languages (and set espeak below)
ESPEAK = 'ur'       # ur=Urdu, pa=Punjabi, ps=Pashto, sd=Sindhi
os.chdir(WORK)
if not KAGGLE and not os.path.isdir('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
# locate dataset: a zip OR an already-extracted folder (Kaggle unzips uploads)
metas = glob.glob('/kaggle/input/**/metadata.csv', recursive=True) + glob.glob('/content/drive/MyDrive/eyewaz/**/metadata.csv', recursive=True)
zips  = glob.glob('/kaggle/input/**/dataset-*.zip', recursive=True) + glob.glob('/content/drive/MyDrive/eyewaz/dataset-*.zip')
assert metas or zips, 'No dataset. Kaggle: Add Input > Upload dataset-male.zip. Colab: put it in Drive/eyewaz/'
# system deps + clone the maintained Piper trainer
!apt-get -qq install -y espeak-ng libespeak-ng-dev build-essential cmake ninja-build >/dev/null 2>&1
if not os.path.exists(WORK+'/piper1'):
    !git clone -q https://github.com/OHF-Voice/piper1-gpl {WORK}/piper1
os.chdir(WORK+'/piper1')
# install: non-editable .[train] compiles espeakbridge AND pulls lightning/pysilero/etc.
!pip install -q '.[train]' 2>&1 | tail -5
pdir = subprocess.run(['python3','-c','import piper,os;print(os.path.dirname(piper.__file__))'],capture_output=True,text=True).stdout.strip()
so = glob.glob(os.path.join(pdir,'espeakbridge*.so')); assert so, f'espeakbridge not built in {pdir}'
shutil.copy(so[0], WORK+'/eb.so')
# editable install exposes the vits training source; copy the compiled bridge in
!pip install -q -e . --force-reinstall --no-deps
shutil.copy(WORK+'/eb.so', WORK+'/piper1/src/piper/'+os.path.basename(so[0]))
!bash build_monotonic_align.sh
# prepare dataset -> wav|text CSV
os.chdir(WORK)
if metas: DATA = os.path.dirname(metas[0])
else:
    !unzip -oq {zips[0]} -d {WORK}/
    DATA = sorted(glob.glob(WORK+'/dataset-*'))[0]
AUDIO = f'{DATA}/wav'; CSV = f'{WORK}/{VOICE}.csv'
with open(f'{DATA}/metadata.csv',encoding='utf-8') as f, open(CSV,'w',encoding='utf-8') as o:
    for line in f:
        line=line.strip()
        if '|' in line:
            cid,t=line.split('|',1); o.write(f'{cid}.wav|{t}\n')
PT = f'{WORK}/pretrained.ckpt'
if not os.path.exists(PT):
    !wget -q -O {PT} 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
!python3 -c "from piper import espeakbridge; from piper.train.vits.dataset import VitsDataModule; print('--- imports OK ---')"
print('SETUP DONE | clips:', len(os.listdir(AUDIO)), '| audio:', AUDIO, '| pretrained MB:', round(os.path.getsize(PT)/1e6,1))


## 2 · Train  (watch live `Epoch N … loss=…`; ■ stop after 30+ min, then run cell 3)


In [ ]:
import torch, os, glob
KAGGLE = os.path.isdir('/kaggle/working'); WORK = '/kaggle/working' if KAGGLE else '/content'
VOICE='male'; ESPEAK='ur'; OUT=f'{WORK}/out'; CSV=f'{WORK}/{VOICE}.csv'
metas = glob.glob('/kaggle/input/**/metadata.csv', recursive=True) + glob.glob(f'{WORK}/dataset-*/metadata.csv') + glob.glob('/content/drive/MyDrive/eyewaz/**/metadata.csv', recursive=True)
AUDIO = os.path.dirname(metas[0])+'/wav'
# weights_only patch (PyTorch 2.6) for the training subprocess
open(WORK+'/piper1/sitecustomize.py','w').write("import torch\n_o=torch.load\ndef _p(*a,**k):\n k['weights_only']=False\n return _o(*a,**k)\ntorch.load=_p\n")
# resume from the latest checkpoint, else warm-start from Lessac (weights compatible)
latest = sorted(glob.glob(f'{OUT}/**/*.ckpt', recursive=True), key=os.path.getmtime)
if latest:
    RESUME = latest[-1]; print('resuming from', RESUME)
else:
    ck = torch.load(f'{WORK}/pretrained.ckpt', map_location='cpu', weights_only=False)
    for k in ['hyper_parameters','hparams_name','datamodule_hyper_parameters','loops']: ck.pop(k, None)
    ck['epoch']=0; ck['global_step']=0
    for s in (ck.get('lr_schedulers') or []):
        if isinstance(s, dict): s['last_epoch']=0; s['_step_count']=1
    torch.save(ck, f'{WORK}/warm.ckpt'); RESUME=f'{WORK}/warm.ckpt'; print('warm-start from Lessac')
os.chdir(WORK+'/piper1')
!PYTHONPATH={WORK}/piper1 python3 -m piper.train fit \
  --data.voice_name eyewaz-urdu-{VOICE} \
  --data.csv_path {CSV} \
  --data.audio_dir {AUDIO} \
  --model.sample_rate 22050 \
  --data.espeak_voice {ESPEAK} \
  --data.cache_dir {WORK}/cache \
  --data.config_path {WORK}/eyewaz-urdu-{VOICE}.json \
  --data.batch_size 16 \
  --ckpt_path {RESUME} \
  --trainer.max_epochs 1000 \
  --trainer.default_root_dir {OUT} \
  --trainer.accelerator gpu --trainer.devices 1


## 3 · Export to ONNX  (+ download / Save Version)


In [ ]:
import glob, os
KAGGLE = os.path.isdir('/kaggle/working'); WORK = '/kaggle/working' if KAGGLE else '/content'; VOICE='male'
cks = sorted(glob.glob(f'{WORK}/out/**/*.ckpt', recursive=True), key=os.path.getmtime)
assert cks, 'No checkpoint yet — let the TRAIN cell run more epochs.'
ck = cks[-1]; print('exporting', ck)
# run export from inside piper1 so the weights_only patch (sitecustomize) applies
!cd {WORK}/piper1 && PYTHONPATH={WORK}/piper1 python3 -m piper.train.export_onnx --checkpoint {ck} --output-file {WORK}/eyewaz-urdu-{VOICE}.onnx
!cp {WORK}/eyewaz-urdu-{VOICE}.json {WORK}/eyewaz-urdu-{VOICE}.onnx.json
mb = round(os.path.getsize(f'{WORK}/eyewaz-urdu-{VOICE}.onnx')/1e6, 1)
print(f'EXPORTED eyewaz-urdu-{VOICE}.onnx  ({mb} MB)  + .onnx.json')
if KAGGLE:
    print('>>> Click SAVE VERSION (Quick Save) to keep these, then download both from the Output panel.')
else:
    from google.colab import files
    files.download(f'{WORK}/eyewaz-urdu-{VOICE}.onnx'); files.download(f'{WORK}/eyewaz-urdu-{VOICE}.onnx.json')


### Send me `eyewaz-urdu-male.onnx` **and** `eyewaz-urdu-male.onnx.json` and I wire your voice into every EYEWAZ surface.
